# **EDA BRONZE ANALYSIS**

In [0]:
%sql
DESCRIBE TABLE EXTENDED products.bronze.product_list_markets_bronze

In [0]:
%sql
select count(*) from products.bronze.product_list_markets_bronze

In [0]:
%sql

--Count Null Values per Column

select
  count(*) - count(nombre_mercado) as nombre_mercado_null,
  count(*) - count(product_list_id) as product_list_id_null,
  count(*) - count(unidad_medida) as unidad_medida_null,
  count(*) - count(inicio_fecha_precios) as inicio_fecha_precios_null,
  count(*) - count(fin_fecha_precios) as fin_fecha_precios_null,
  count(*) - count(precio_inicio) as precio_inicio_null,
  count(*) - count(precio_fin) as precio_fin_null,
  count(*) - count(error) as error_null
from
  products.bronze.product_list_markets_bronze

In [0]:
%sql
-- Null Values Percentage

with total_values as(
    select count(*) as total from products.bronze.product_list_markets_bronze
)

select
  round((tt.total - count(nombre_mercado)) * 100.0 / tt.total, 2) as nombre_mercado_null_pct,
  round((tt.total - count(product_list_id)) * 100.0 / tt.total, 2) as product_list_id_null_pct,
  round((tt.total - count(unidad_medida)) * 100.0 / tt.total, 2) as unidad_medida_null_pct,
  round((tt.total - count(inicio_fecha_precios)) * 100.0 / tt.total, 2) as inicio_fecha_precios_null_pct,
  round((tt.total - count(fin_fecha_precios)) * 100.0 / tt.total, 2) as fin_fecha_precios_null_pct,
  round((tt.total - count(precio_inicio)) * 100.0 / tt.total, 2) as precio_inicio_null_pct,
  round((tt.total - count(precio_fin)) * 100.0 / tt.total, 2) as precio_fin_null_pct
from
  products.bronze.product_list_markets_bronze
cross join total_values tt
group by tt.total

In [0]:
%sql

-- Percentages Market Distribution

select distinct
  nombre_mercado,
  count(*) as total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as porcentage
from
  products.bronze.product_list_markets_bronze
group by nombre_mercado
order by total desc

In [0]:
%sql
-- Percentages Product Distribution

select distinct
  nombre_producto,
  count(*) as total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as porcentage
from
  products.bronze.product_list_markets_bronze
group by nombre_producto
order by total desc

In [0]:
%sql
-- Percentages Measurement Distribution

select distinct
  unidad_medida,
  count(*) as total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as porcentage
from
  products.bronze.product_list_markets_bronze
group by unidad_medida
order by total desc

In [0]:
%sql

create or replace temporary view products_bronze_transform_analysis as
select
  nombre_mercado,
  nombre_producto,
  unidad_medida,
  case
    when
      precio_inicio rlike '^-?[0-9]+\\.?[0-9]*$'
      and precio_inicio::double between 0 and 2147483648
    then
      precio_inicio::double
    else null
  end as precio_inicio_clean,
  case
    when
      precio_fin rlike '^-?[0-9]+\\.?[0-9]*$'
      and precio_fin::double between 0 and 2147483648
    then
      precio_fin::double
    else null
  end as precio_fin_clean,
  case
    when inicio_fecha_precios RLIKE '^[^a-zA-Z]+$' then to_date(inicio_fecha_precios, 'dd-MM-yyyy')
    else null
  end as inicio_fecha_precios_clean,
  case
    when fin_fecha_precios RLIKE '^[^a-zA-Z]+$' then to_date(fin_fecha_precios, 'dd-MM-yyyy')
    else null
  end as fin_fecha_precios_clean
from
  products.bronze.product_list_markets_bronze

In [0]:
%sql
select
  max(precio_inicio_clean) as max_precio_inicio_clean,
  min(precio_inicio_clean) as min_precio_inicio_clean,
  round(avg(precio_inicio_clean), 2) as avg_precio_inicio_clean,
  max(precio_fin_clean) as max_precio_fin_clean,
  min(precio_fin_clean) as min_precio_fin_clean,
  round(avg(precio_fin_clean), 2) as avg_precio_fin_clean,
  max(inicio_fecha_precios_clean) as max_inicio_fecha_precios_clean,
  min(inicio_fecha_precios_clean) as min_inicio_fecha_precios_clean,
  max(fin_fecha_precios_clean) as max_fin_fecha_precios_clean,
  min(fin_fecha_precios_clean) as min_fin_fecha_precios_clean
from
  products_bronze_transform_analysis